# Normalizacao do Relatorio de Receitas do Sistema ATUA

Este notebook le o arquivo `Relatorio_Receitas_Sistema-ATUA_{MM}.xls` (sistema ATUA, GSL Logistica) e converte para o layout do fechamento SAGI (`FECHAMENTO_ODBC_{AAAA}_{MM}.xlsx`).

Ajuste o parametro `MES_REFERENCIA` na primeira celula de codigo (formato `MM/AAAA`).

Cada linha do arquivo e um CTRC (Conhecimento de Transporte Rodoviario de Cargas) — ou seja, uma receita de frete faturada. No SAGI, a GSL esta na divisao **2.4 TRANSMOVE** (lado receita), com tres filiais ativas neste relatorio:

- GSL PRUDENTE -> 2.4.1 PRESIDENTE PRUDENTE -> 2.4.1.1 TRANSPORTE
- GSL DOURADOS -> 2.4.2 DOURADOS -> 2.4.2.1 TRANSPORTE
- GSL MARINGA PR -> 2.4.3 MARINGA -> 2.4.3.1 TRANSPORTE

O Plano de Contas de todas as linhas e **5.7.1 FRETES** (receita de frete proprio).

In [7]:
from pathlib import Path
import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

# ── Parametros ──────────────────────────────────────────────────────────────
MES_REFERENCIA = "04/2026"   # formato MM/AAAA — ajuste para cada fechamento
# ────────────────────────────────────────────────────────────────────────────

mes_num, ano = MES_REFERENCIA.split("/")

REFS_DIR = Path("../../02-Referencias")
ATUA_DIR = REFS_DIR / "ATUA"
ARQUIVO_ENTRADA = ATUA_DIR / f"Relatorio_Receitas_Sistema-ATUA_{mes_num}.xls"
ARQUIVO_MODELO  = REFS_DIR / f"FECHAMENTO_ODBC_{ano}_{mes_num}.xlsx"
ARQUIVO_SAIDA   = ATUA_DIR / f"ATUA_receitas_fechamento_{mes_num}-{ano}.xlsx"

if not ARQUIVO_ENTRADA.exists():
    raise FileNotFoundError(f"Arquivo ATUA Receitas nao encontrado: {ARQUIVO_ENTRADA.resolve()}")

# O arquivo possui pequena corrupcao interna — lemos com ignore_workbook_corruption
df_receitas = pd.read_excel(
    ARQUIVO_ENTRADA,
    sheet_name=0,
    header=0,
    dtype=object,
    engine_kwargs={"ignore_workbook_corruption": True},
)

print(f"Entrada: {ARQUIVO_ENTRADA.resolve()}")
print(f"Linhas lidas: {len(df_receitas)}")
print(f"Colunas: {df_receitas.columns.tolist()}")
print()
df_receitas.head(5)

Entrada: C:\Users\julio.santana\Documents\Projects\Cofre_Trabalho\02-Referencias\ATUA\Relatorio_Receitas_Sistema-ATUA_04.xls
Linhas lidas: 200
Colunas: ['cd_ctrc', 'nr_ctrc', 'ds_serie', 'nr_cfop', 'id_sintegra', 'ds_servico', 'id_situacao_cte', 'nr_contrato', 'nr_pedido', 'nr_ordem_compra', 'nr_liberacao_embarque', 'dt_cancelamento', 'dt_emissao', 'dt_mes_emissao', 'dt_lancamento', 'dt_chegada', 'cd_pessoa_filial', 'nm_pessoa_filial', 'cd_agencia', 'nm_agencia', 'nm_cidade_origem', 'ds_uf_origem', 'nm_cidade_destino', 'ds_uf_destino', 'cd_pessoa_usuario', 'nm_pessoa_usuario', 'cd_pessoa_embarque', 'nm_pessoa_remetente', 'nr_cnpj_cpf_remetente', 'ds_endereco_remetente', 'ds_uf_remetente', 'nm_cidade_remetente', 'cd_pessoa_consignatario', 'nm_pessoa_consignatario', 'nr_cnpj_cpf_consignatario', 'ds_uf_consignatario', 'nm_cidade_consignatario', 'cd_pessoa_consignatario_embarque', 'nm_pessoa_consignatario_embarque', 'cd_pessoa_entrega', 'nm_pessoa_destinatario', 'nr_cnpj_cpf_destinatario',

,cd_ctrc,nr_ctrc,ds_serie,nr_cfop,id_sintegra,ds_servico,id_situacao_cte,nr_contrato,nr_pedido,nr_ordem_compra,nr_liberacao_embarque,dt_cancelamento,dt_emissao,dt_mes_emissao,dt_lancamento,...,id_issqn_retido,dt_embarque,dt_chegada_destino,nr_solicitacao,nr_lacres,pr_desconto_base_comissao_ctrc,pr_desconto_mercadoria_base_comissao_ctrc,id_proprietario_veiculo,ds_operacao_minuta,nr_nfse,id_pendente_emissao_nfse,perna,nr_ctrc_primeira_perna,vl_ibs,vl_cbs
0,25589,247,2,6353,CTRC,Normal,Autorizado,NaN,NaN,NaN,NaN,NaN,01/04/26,04/2026,2026-04-01 09:34:11,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Terceiro,NaN,NaN,NaN,0,NaN,2,17.97
1,25590,248,2,6353,CTRC,Normal,Autorizado,NaN,NaN,NaN,NaN,NaN,01/04/26,04/2026,2026-04-01 09:34:11,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Terceiro,NaN,NaN,NaN,0,NaN,1.92,17.25
2,25591,908,1,5352,CTRC,Normal,Autorizado,NaN,NaN,NaN,NaN,NaN,01/04/26,04/2026,2026-04-01 09:34:11,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Terceiro,NaN,NaN,NaN,0,NaN,7.37,66.36
3,25592,909,1,5352,CTRC,Normal,Autorizado,NaN,NaN,NaN,NaN,NaN,01/04/26,04/2026,2026-04-01 09:34:11,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Terceiro,NaN,NaN,NaN,0,NaN,8.8,79.2
4,25593,910,1,5352,CTRC,Normal,Autorizado,NaN,NaN,NaN,NaN,NaN,01/04/26,04/2026,2026-04-01 09:34:11,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Terceiro,NaN,NaN,NaN,0,NaN,0.48,4.31


## Mapeamento de Centro de Custo (Receita)

O campo `nm_pessoa_filial` no ATUA identifica qual filial GSL emitiu o frete. O mapeamento para a hierarquia SAGI e:

| nm_pessoa_filial | n2     | n3     | n4       | Descricoes                                 |
|------------------|--------|--------|----------|--------------------------------------------|
| GSL PRUDENTE     | 2.4    | 2.4.1  | 2.4.1.1  | TRANSMOVE GSL > PRESIDENTE PRUDENTE > TRANSPORTE |
| GSL DOURADOS     | 2.4    | 2.4.2  | 2.4.2.1  | TRANSMOVE GSL > DOURADOS > TRANSPORTE      |
| GSL MARINGA PR   | 2.4    | 2.4.3  | 2.4.3.1  | TRANSMOVE GSL > MARINGA > TRANSPORTE       |

In [8]:
def _str(v) -> str:
    if pd.isna(v):
        return ""
    return str(v).strip()

MAPA_FILIAL_RECEITA = {
    "GSL PRUDENTE": {
        "n3_cod":  "2.4.1",
        "n3_desc": "PRESIDENTE PRUDENTE",
        "n4_cod":  "2.4.1.1",
        "n4_desc": "TRANSPORTE",
        "filial_saida": "GSL PRUDENTE",
    },
    "GSL DOURADOS": {
        "n3_cod":  "2.4.2",
        "n3_desc": "DOURADOS",
        "n4_cod":  "2.4.2.1",
        "n4_desc": "TRANSPORTE",
        "filial_saida": "GSL DOURADOS",
    },
    "GSL MARINGA PR": {
        "n3_cod":  "2.4.3",
        "n3_desc": "MARINGA",
        "n4_cod":  "2.4.3.1",
        "n4_desc": "TRANSPORTE",
        "filial_saida": "GSL MARINGA",
    },
}

def mapear_cc_receita(nm_filial) -> dict | None:
    """Retorna hierarquia SAGI completa para a filial, ou None se nao mapeada.

    Hierarquia de 3 niveis uteis (o prefixo numerico nao conta como nivel):
      n1: 2.4       -> TRANSMOVE GSL
      n2: 2.4.X     -> filial (ex: 2.4.1 PRESIDENTE PRUDENTE)
      n3: 2.4.X.Y   -> setor  (ex: 2.4.1.1 TRANSPORTE)
      n4: igual a n3 -> padrao do modelo FECHAMENTO_ODBC
    """
    chave = _str(nm_filial)
    info  = MAPA_FILIAL_RECEITA.get(chave)
    if not info:
        return None
    return {
        "n1_cod":  "2.4",
        "n1_desc": "TRANSMOVE GSL",
        "n2_cod":  info["n3_cod"],
        "n2_desc": info["n3_desc"],
        "n3_cod":  info["n4_cod"],
        "n3_desc": info["n4_desc"],
        "n4_cod":  info["n4_cod"],
        "n4_desc": info["n4_desc"],
        "filial_saida": info["filial_saida"],
        "segmento": "TRANSMOVE GSL",
    }

print("Filiais unicas no arquivo:")
print(df_receitas["nm_pessoa_filial"].value_counts(dropna=False).to_string())
print()

filiais_nao_mapeadas = set()
for v in df_receitas["nm_pessoa_filial"].unique():
    chave = _str(v)
    if chave not in MAPA_FILIAL_RECEITA:
        filiais_nao_mapeadas.add(chave)

if filiais_nao_mapeadas:
    print(f"[AVISO] Filiais NAO mapeadas ({len(filiais_nao_mapeadas)}):")
    for f in sorted(filiais_nao_mapeadas):
        print(f"  '{f}'")
else:
    print("[OK] Todas as filiais estao mapeadas.")

Filiais unicas no arquivo:
nm_pessoa_filial
GSL PRUDENTE      118
GSL MARINGA PR     62
GSL DOURADOS       20

[OK] Todas as filiais estao mapeadas.


## Plano de Contas — Fixo: 5.7.1 FRETES

Todas as 138 linhas deste relatorio sao CTRCs (fretes proprios da GSL faturados a clientes). No SAGI, a conta de receita correspondente e **5.7.1 FRETES**.

In [9]:
COD_CONTA_RECEITA  = "5.7.1"
DESC_CONTA_RECEITA = "FRETES"

print(f"Plano de Contas fixo para todas as linhas: {COD_CONTA_RECEITA} {DESC_CONTA_RECEITA}")

Plano de Contas fixo para todas as linhas: 5.7.1 FRETES


## Conversao para o layout FECHAMENTO_ODBC

Regras de mapeamento de colunas:

| Coluna FECHAMENTO_ODBC | Origem ATUA                          | Observacao                                    |
|------------------------|--------------------------------------|-----------------------------------------------|
| filial                 | filial_saida (do mapa CC)            |                                               |
| titulo                 | `CTRC-{nr_ctrc}`                     |                                               |
| credor_forn_cli_func   | nm_pessoa_destinatario               | cliente / tomador do frete                    |
| data_nf                | dt_emissao                           | data de emissao do CTRC                       |
| data_pagamento         | dt_emissao                           | mesma data (nao ha data de pagamento separada) |
| valor_nf               | vl_frete_empresa                     |                                               |
| valor_pago             | vl_frete_empresa                     |                                               |
| valor_conta            | vl_frete_empresa                     |                                               |
| cod_conta              | `5.7.1`                              |                                               |
| conta                  | `FRETES`                             |                                               |
| observacao             | CTRC {nr_ctrc} - Motorista: ... - Destino: ... |                                      |
| Origem                 | `Saida (Aplicacoes)`                 |                                               |
| Sistema                | `ATUA`                               |                                               |
| n1_cod … n4_desc       | MAPA_FILIAL_RECEITA                  |                                               |

In [10]:
def _to_float(v):
    if pd.isna(v):
        return 0.0
    try:
        return float(str(v).strip().replace(",", "."))
    except (ValueError, TypeError):
        return 0.0

def _fmt_brl(v) -> str:
    """Formata float como string no padrao brasileiro (ex: 1.234,56)."""
    try:
        f = _to_float(v)
        return f"{f:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")
    except Exception:
        return str(v)

def _fmt_data(v) -> str:
    """Converte 'dd/mm/yy ' (com possivel espaco) para 'dd/mm/aaaa'."""
    s = _str(v)
    if not s:
        return ""
    # Tenta interpretar formatos possiveis
    for fmt in ("%d/%m/%y", "%d/%m/%Y", "%Y-%m-%d %H:%M:%S", "%Y-%m-%d"):
        try:
            import datetime
            return datetime.datetime.strptime(s, fmt).strftime("%d/%m/%Y")
        except ValueError:
            pass
    return s  # fallback: devolve como veio

# Carrega colunas do modelo para garantir ordem e completude
modelo_cols = pd.read_excel(ARQUIVO_MODELO, nrows=0).columns.tolist()

filiais_sem_mapa = []
linhas_saida = []

for _, row in df_receitas.iterrows():
    filial_raw = _str(row.get("nm_pessoa_filial", ""))
    cc = mapear_cc_receita(filial_raw)

    if cc is None:
        filiais_sem_mapa.append(filial_raw)

    ctrc     = _str(row.get("nr_ctrc", ""))
    dt_emis  = _fmt_data(row.get("dt_emissao", ""))
    valor    = _to_float(row.get("vl_frete_empresa", 0))
    destino  = _str(row.get("nm_cidade_destinatario", ""))
    motorist = _str(row.get("nm_pessoa_motorista", ""))
    cliente  = _str(row.get("nm_pessoa_destinatario", ""))

    nova = {col: "" for col in modelo_cols}

    nova["filial"]               = cc["filial_saida"] if cc else filial_raw
    nova["titulo"]               = str(ctrc)
    nova["credor_forn_cli_func"] = cliente
    nova["data_nf"]              = dt_emis
    nova["data_pagamento"]       = dt_emis
    nova["valor_nf"]             = _fmt_brl(valor)
    nova["valor_pago"]           = _fmt_brl(valor)
    nova["valor_conta"]          = _fmt_brl(valor)
    nova["cod_conta"]            = COD_CONTA_RECEITA
    nova["conta"]                = DESC_CONTA_RECEITA
    nova["observacao"]           = f"CTRC {ctrc} - Motorista: {motorist} - Destino: {destino}"
    nova["Origem"]               = "Saida (Aplicacoes)"
    nova["Sistema"]              = "ATUA"
    nova["Segmento"]             = cc["segmento"]     if cc else ""
    nova["cod_conta-descr"]      = f"{COD_CONTA_RECEITA} {DESC_CONTA_RECEITA}"
    nova["n1_cod_centro_custo"]  = cc["n1_cod"]       if cc else ""
    nova["n1_centro_custo"]      = cc["n1_desc"]      if cc else ""
    nova["n1_CC"]                = f"{cc['n1_cod']} {cc['n1_desc']}" if cc else ""
    nova["n2_cod_centro_custo"]  = cc["n2_cod"]       if cc else ""
    nova["n2_centro_custo"]      = cc["n2_desc"]      if cc else ""
    nova["n2_CC"]                = f"{cc['n2_cod']} {cc['n2_desc']}" if cc else ""
    nova["n3_cod_centro_custo"]  = cc["n3_cod"]       if cc else ""
    nova["n3_centro_custo"]      = cc["n3_desc"]      if cc else ""
    nova["n3_CC"]                = f"{cc['n3_cod']} {cc['n3_desc']}" if cc else ""
    nova["n4_cod_centro_custo"]  = cc["n4_cod"]       if cc else ""
    nova["n4_centro_custo"]      = cc["n4_desc"]      if cc else ""
    nova["n4_CC"]                = f"{cc['n4_cod']} {cc['n4_desc']}" if cc else ""

    linhas_saida.append(nova)

fechamento_df = pd.DataFrame(linhas_saida, columns=modelo_cols)

print(f"Linhas geradas: {len(fechamento_df)}")
print(f"Filiais sem mapeamento: {len(filiais_sem_mapa)}")
if filiais_sem_mapa:
    print(set(filiais_sem_mapa))
print()
fechamento_df.head(5)

Linhas geradas: 200
Filiais sem mapeamento: 0



,id,Segmento,n1_cod_centro_custo,n1_centro_custo,n1_CC,n2_cod_centro_custo,n2_centro_custo,n2_CC,n3_cod_centro_custo,n3_centro_custo,n3_CC,n4_cod_centro_custo,n4_centro_custo,n4_CC,cod_conta,...,Unnamed: 38,Unnamed: 39,Unnamed: 40,Unnamed: 41,Unnamed: 42,Unnamed: 43,Unnamed: 44,Unnamed: 45,Unnamed: 46,Unnamed: 47,Unnamed: 48,Unnamed: 49,Unnamed: 50,Unnamed: 51,Unnamed: 52
0,,TRANSMOVE GSL,2.4,TRANSMOVE GSL,2.4 TRANSMOVE GSL,2.4.3,MARINGA,2.4.3 MARINGA,2.4.3.1,TRANSPORTE,2.4.3.1 TRANSPORTE,2.4.3.1,TRANSPORTE,2.4.3.1 TRANSPORTE,5.7.1,...,,,,,,,,,,,,,,,
1,,TRANSMOVE GSL,2.4,TRANSMOVE GSL,2.4 TRANSMOVE GSL,2.4.3,MARINGA,2.4.3 MARINGA,2.4.3.1,TRANSPORTE,2.4.3.1 TRANSPORTE,2.4.3.1,TRANSPORTE,2.4.3.1 TRANSPORTE,5.7.1,...,,,,,,,,,,,,,,,
2,,TRANSMOVE GSL,2.4,TRANSMOVE GSL,2.4 TRANSMOVE GSL,2.4.1,PRESIDENTE PRUDENTE,2.4.1 PRESIDENTE PRUDENTE,2.4.1.1,TRANSPORTE,2.4.1.1 TRANSPORTE,2.4.1.1,TRANSPORTE,2.4.1.1 TRANSPORTE,5.7.1,...,,,,,,,,,,,,,,,
3,,TRANSMOVE GSL,2.4,TRANSMOVE GSL,2.4 TRANSMOVE GSL,2.4.1,PRESIDENTE PRUDENTE,2.4.1 PRESIDENTE PRUDENTE,2.4.1.1,TRANSPORTE,2.4.1.1 TRANSPORTE,2.4.1.1,TRANSPORTE,2.4.1.1 TRANSPORTE,5.7.1,...,,,,,,,,,,,,,,,
4,,TRANSMOVE GSL,2.4,TRANSMOVE GSL,2.4 TRANSMOVE GSL,2.4.1,PRESIDENTE PRUDENTE,2.4.1 PRESIDENTE PRUDENTE,2.4.1.1,TRANSPORTE,2.4.1.1 TRANSPORTE,2.4.1.1,TRANSPORTE,2.4.1.1 TRANSPORTE,5.7.1,...,,,,,,,,,,,,,,,


## Salvando o arquivo de saida

In [11]:
from openpyxl.styles import Font

arquivo_saida_exec = ARQUIVO_SAIDA

try:
    with pd.ExcelWriter(arquivo_saida_exec, engine="openpyxl") as writer:
        sheet_name = "ATUA_receitas"
        fechamento_df.to_excel(writer, sheet_name=sheet_name, index=False)
        ws = writer.book[sheet_name]
        bold_font = Font(bold=True)
        for cell in ws[1]:
            cell.font = bold_font
    print(f"Arquivo gerado: {arquivo_saida_exec.resolve()}")
    print(f"Linhas gravadas: {len(fechamento_df)}")
except PermissionError:
    import datetime
    ts = datetime.datetime.now().strftime("%H%M%S")
    alt = arquivo_saida_exec.with_stem(f"{arquivo_saida_exec.stem}_{ts}")
    with pd.ExcelWriter(alt, engine="openpyxl") as writer:
        sheet_name = "ATUA_receitas"
        fechamento_df.to_excel(writer, sheet_name=sheet_name, index=False)
        ws = writer.book[sheet_name]
        bold_font = Font(bold=True)
        for cell in ws[1]:
            cell.font = bold_font
    print(f"[AVISO] Arquivo principal em uso. Salvo como: {alt.resolve()}")
    print(f"Linhas gravadas: {len(fechamento_df)}")

print()

if filiais_sem_mapa:
    print(f"[PENDENTE] {len(filiais_sem_mapa)} linha(s) com filial nao mapeada:")
    for f in sorted(set(filiais_sem_mapa)):
        n = filiais_sem_mapa.count(f)
        print(f"  '{f}' ({n} ocorrencia(s))")
else:
    print("[OK] Todas as filiais foram mapeadas. Nenhum item pendente.")

Arquivo gerado: C:\Users\julio.santana\Documents\Projects\Cofre_Trabalho\02-Referencias\ATUA\ATUA_receitas_fechamento_04-2026.xlsx
Linhas gravadas: 200

[OK] Todas as filiais foram mapeadas. Nenhum item pendente.
